In [1]:
# Load env variables and create client
from dotenv import load_dotenv
from anthropic import Anthropic

load_dotenv()

client = Anthropic()
model = "claude-haiku-4-5"

In [2]:
# Helper functions
def add_user_message(messages, text):
    user_message = {"role": "user", "content": text}
    messages.append(user_message)


def add_assistant_message(messages, text):
    assistant_message = {"role": "assistant", "content": text}
    messages.append(assistant_message)


def chat(messages, system=None, temperature=1.0, stop_sequences=[]):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": temperature,
        "stop_sequences": stop_sequences,
    }

    if system:
        params["system"] = system

    message = client.messages.create(**params)
    return message.content[0].text

In [3]:
# Function to generate a new dataset
import json


def generate_dataset():
    prompt = """
Generate a evaluation dataset for a prompt evaluation. The dataset will be used to evaluate prompts
that generate Python, JSON, or Regex specifically for AWS-related tasks. Generate an array of JSON objects,
each representing task that requires Python, JSON, or a Regex to complete.

Example output:
```json
[
    {
        "task": "Description of task",
        "format": "json" or "python" or "regex"
    },
    ...additional
]
```

* Focus on tasks that can be solved by writing a single Python function, a single JSON object, or a regular expression.
* Focus on tasks that do not require writing much code

Please generate 3 objects.
"""

    messages = []
    add_user_message(messages, prompt)
    add_assistant_message(messages, "```json")
    text = chat(messages, stop_sequences=["```"])
    return json.loads(text)

In [4]:
# Generate the dataset and write it to 'dataset.json'
dataset = generate_dataset()
with open("dataset.json", "w") as f:
    json.dump(dataset, f, indent=2)

In [5]:
# Function to grade a test case + output using a model
import re

def grade_by_model(test_case, output):
    eval_prompt = f"""
You are an expert AWS code reviewer. Your task is to evaluate the following AI-generated solution.

Original Task:
<task>
{test_case["task"]}
</task>

Solution to Evaluate:
<solution>
{output}
</solution>

Output Format
Provide your evaluation as a structured JSON object with the following fields, in this specific order:
- "strengths": An array of 1-3 key strengths
- "weaknesses": An array of 1-3 key areas for improvement
- "reasoning": A concise explanation of your overall assessment
- "score": A number between 1-10

Respond with JSON. Keep your response concise and direct.
Example response shape:
{{
    "strengths": string[],
    "weaknesses": string[],
    "reasoning": string,
    "score": number
}}
    """

    messages = []
    add_user_message(messages, eval_prompt)
    add_assistant_message(messages, "```json")
    eval_text = chat(messages, stop_sequences=["```"])
    # Fix invalid JSON escape sequences (e.g. \s, \e from code snippets in reasoning)
    eval_text_fixed = re.sub(r'\\(?!["\\/bfnrtu])', r'\\\\', eval_text)
    return json.loads(eval_text_fixed)

In [6]:
# Passes a test case into Claude
def run_prompt(test_case):
    prompt = f"""
Please solve the following task:

{test_case["task"]}
"""

    messages = []
    add_user_message(messages, prompt)
    output = chat(messages)
    return output

In [7]:
# Function to execute a single test case and grade the output
def run_test_case(test_case):
    """Calls run_prompt, then grades the result"""
    output = run_prompt(test_case)

    model_grade = grade_by_model(test_case, output)
    score = model_grade["score"]
    reasoning = model_grade["reasoning"]

    return {
        "output": output,
        "test_case": test_case,
        "score": score,
        "reasoning": reasoning
    }

In [8]:
def run_eval(dataset):
    """Loads the dataset and calls run_test_case with each case"""
    results = []
    
    for test_case in dataset:
        result = run_test_case(test_case)
        results.append(result)
    
    return results

In [9]:
with open("dataset.json", "r") as f:
    dataset = json.load(f)

results = run_eval(dataset)

In [10]:
print(json.dumps(results, indent=2))

[
  {
    "output": "# AWS Lambda Function Configuration for S3 Events\n\n```json\n{\n  \"FunctionName\": \"S3EventProcessor\",\n  \"Runtime\": \"python3.11\",\n  \"Role\": \"arn:aws:iam::ACCOUNT_ID:role/lambda-s3-execution-role\",\n  \"Handler\": \"index.lambda_handler\",\n  \"Timeout\": 300,\n  \"MemorySize\": 512,\n  \"Description\": \"Processes S3 bucket events with 5-minute timeout and 512MB memory\",\n  \"Environment\": {\n    \"Variables\": {\n      \"LOG_LEVEL\": \"INFO\"\n    }\n  },\n  \"VpcConfig\": {\n    \"SecurityGroupIds\": [],\n    \"SubnetIds\": []\n  },\n  \"Tags\": {\n    \"Environment\": \"Production\",\n    \"Application\": \"S3EventProcessor\"\n  },\n  \"EphemeralStorage\": {\n    \"Size\": 512\n  },\n  \"Architectures\": [\n    \"x86_64\"\n  ],\n  \"EventInvokeConfig\": {\n    \"MaximumEventAge\": 3600,\n    \"MaximumRetryAttempts\": 2\n  }\n}\n```\n\n## S3 Event Source Configuration\n\n```json\n{\n  \"EventSourceMappings\": [\n    {\n      \"EventSource\": \"s3:

In [8]:
from statistics import mean


def run_eval(dataset):
    """Loads the dataset and calls run_test_case with each case"""
    results = []

    for test_case in dataset:
        result = run_test_case(test_case)
        results.append(result)

    average_score = mean([result["score"] for result in results])
    print(f"Average score: {average_score}")

    return results

In [9]:
with open("dataset.json", "r") as f:
    dataset = json.load(f)

results = run_eval(dataset)

Average score: 7


In [10]:
print(json.dumps(results, indent=2))

[
  {
    "output": "# AWS S3 Bucket Name Parser\n\nHere's a solution to parse an S3 bucket name from an S3 URI:\n\n```python\ndef parse_s3_bucket_name(s3_uri: str) -> str:\n    \"\"\"\n    Extract the bucket name from an S3 URI.\n    \n    Args:\n        s3_uri: S3 URI in format s3://bucket-name/key/path\n        \n    Returns:\n        The bucket name\n        \n    Raises:\n        ValueError: If the URI is not a valid S3 URI\n    \"\"\"\n    if not s3_uri.startswith(\"s3://\"):\n        raise ValueError(\"Invalid S3 URI: must start with 's3://'\")\n    \n    # Remove 's3://' prefix\n    uri_without_prefix = s3_uri[5:]\n    \n    # Split by '/' and get the first part (bucket name)\n    bucket_name = uri_without_prefix.split(\"/\")[0]\n    \n    if not bucket_name:\n        raise ValueError(\"Invalid S3 URI: bucket name is empty\")\n    \n    return bucket_name\n\n\n# Test cases\nif __name__ == \"__main__\":\n    test_cases = [\n        \"s3://my-bucket/key/path\",\n        \"s3://my